# LeetCode #45 — Jump Game II: Greedy theo "tầng" (BFS ngầm)

**Mức độ:** Medium · **Chủ đề:** Array, Greedy, Dynamic Programming, BFS

Skeleton trên LeetCode:

```python
class Solution:
    def jump(self, nums: List[int]) -> int:
```

Mục tiêu của notebook:

1. Giải bằng **Quy hoạch động (DP)** `O(n²)` — cách tự nhiên và chắc chắn đúng.
2. Nhận ra cấu trúc **"tầng"**: các vị trí tới được bằng đúng `k` bước tạo thành một **đoạn liên tiếp**.
3. Từ đó suy ra lời giải **Greedy `O(n)`** chỉ với hai biến `current_end`, `farthest` — thực chất là BFS không cần hàng đợi.

# 1. Bài toán là gì?

Cho mảng `nums` dài `n`, bạn đứng ở chỉ số `0`. `nums[i]` là **độ dài nhảy tối đa** từ vị trí `i`: từ `i` có thể nhảy tới bất kỳ `i + j` với `0 <= j <= nums[i]` và `i + j < n`.

Trả về **số lần nhảy ít nhất** để tới chỉ số `n - 1`. Đề **đảm bảo** luôn tới được.

**Constraints (theo đề):**

- `1 <= nums.length <= 10^4`
- `0 <= nums[i] <= 1000`
- Luôn tới được `nums[n - 1]`.

---

## Ví dụ

```text
nums = [2, 3, 1, 1, 4]  ->  2      (0 -> 1 -> 4)
nums = [2, 3, 0, 1, 4]  ->  2
```

Ví dụ 1: từ `0` nhảy 1 bước tới chỉ số `1` (giá trị 3), rồi nhảy 3 bước tới chỉ số `4`.

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Thuật toán tham lam (greedy)
2. Duyệt theo chiều rộng (BFS — Breadth-First Search)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Thuật toán tham lam (greedy)

### Tham lam là gì?

Thuật toán tham lam xây lời giải bằng cách ở **mỗi bước chọn phương án tốt nhất trước mắt**, không bao giờ quay lại sửa. Nhanh và đơn giản — nhưng **không phải lúc nào cũng đúng**.

### Ví dụ đúng: đổi tiền với mệnh giá 1, 2, 5, 10

Luôn lấy tờ **lớn nhất** không vượt số tiền còn lại.

### Ví dụ sai: mệnh giá 1, 3, 4, cần 6

Tham lam: `4 + 1 + 1` (3 tờ). Tối ưu: `3 + 3` (2 tờ). Chọn tốt nhất trước mắt dẫn tới kết quả tệ hơn.

In [1]:
def doi_tien_tham_lam(menh_gia, so_tien):
    chon = []
    for m in sorted(menh_gia, reverse=True):
        while so_tien >= m:
            so_tien -= m
            chon.append(m)
    return chon

def doi_tien_toi_uu(menh_gia, so_tien):           # DP để đối chiếu
    INF = float("inf")
    dp = [0] + [INF] * so_tien
    for x in range(1, so_tien + 1):
        for m in menh_gia:
            if m <= x:
                dp[x] = min(dp[x], dp[x - m] + 1)
    return dp[so_tien]

print("mệnh giá [1,2,5,10], 18:", doi_tien_tham_lam([1, 2, 5, 10], 18), "| tối ưu:", doi_tien_toi_uu([1, 2, 5, 10], 18), "tờ")
print("mệnh giá [1,3,4], 6:", doi_tien_tham_lam([1, 3, 4], 6), "| tối ưu:", doi_tien_toi_uu([1, 3, 4], 6), "tờ")

mệnh giá [1,2,5,10], 18: [10, 5, 2, 1] | tối ưu: 4 tờ
mệnh giá [1,3,4], 6: [4, 1, 1] | tối ưu: 2 tờ


### Làm sao biết tham lam đúng?

Phải **chứng minh**. Hai kỹ thuật hay dùng:

1. **Lập luận trao đổi (exchange argument)**: giả sử có lời giải tối ưu khác lựa chọn tham lam ở bước đầu; chỉ ra có thể **đổi** nó sang lựa chọn tham lam mà không tệ đi.
2. **Bất biến "luôn dẫn đầu"**: sau mỗi bước, lời giải tham lam không kém bất kỳ lời giải nào khác.

Và luôn **thử phản ví dụ** bằng brute force trên dữ liệu nhỏ.

### Ví dụ kinh điển: chọn nhiều cuộc họp nhất không trùng giờ

Chọn cuộc họp **kết thúc sớm nhất** trước — để lại nhiều thời gian nhất cho phần còn lại (lập luận trao đổi: thay cuộc họp đầu của lời giải tối ưu bằng cuộc kết thúc sớm nhất vẫn không chồng lấn).

In [2]:
def nhieu_cuoc_hop(hop):
    chon, ket_thuc = [], float("-inf")
    for s, e in sorted(hop, key=lambda x: x[1]):     # theo giờ kết thúc
        if s >= ket_thuc:
            chon.append((s, e))
            ket_thuc = e
    return chon

hop_demo = [(1, 4), (3, 5), (0, 6), (5, 7), (3, 9), (5, 9), (6, 10), (8, 11), (8, 12), (2, 14), (12, 16)]
def theo_bat_dau(hop):                               # tiêu chí SAI để so sánh
    chon, ket_thuc = [], float("-inf")
    for s, e in sorted(hop):
        if s >= ket_thuc:
            chon.append((s, e)); ket_thuc = e
    return chon

print("kết thúc sớm nhất trước:", nhieu_cuoc_hop(hop_demo), "→", len(nhieu_cuoc_hop(hop_demo)), "cuộc")
print("bắt đầu sớm nhất trước (sai):", theo_bat_dau(hop_demo), "→", len(theo_bat_dau(hop_demo)), "cuộc")

kết thúc sớm nhất trước: [(1, 4), (5, 7), (8, 11), (12, 16)] → 4 cuộc
bắt đầu sớm nhất trước (sai): [(0, 6), (6, 10), (12, 16)] → 3 cuộc


### Lỗi hay gặp

- Tin tham lam đúng mà không chứng minh / không thử phản ví dụ.
- Chọn sai tiêu chí tham lam (bắt đầu sớm nhất, ngắn nhất… thay vì kết thúc sớm nhất).
- Quên sắp xếp theo tiêu chí trước khi duyệt.

## 2.2 Duyệt theo chiều rộng (BFS — Breadth-First Search)

### BFS là gì?

BFS duyệt đồ thị/cây **theo từng lớp**: thăm điểm xuất phát, rồi **mọi** điểm cách 1 bước, rồi mọi điểm cách 2 bước, … giống vết sóng lan ra khi ném hòn đá xuống nước.

Công cụ: **hàng đợi** (FIFO). Điểm phát hiện trước được xử lý trước → các điểm gần luôn được xử lý trước các điểm xa.

```text
        A            lớp 0: A
       / \           lớp 1: B, C
      B   C          lớp 2: D, E, F
     / \   \
    D   E   F        thứ tự BFS: A B C D E F
```

### Khung BFS trên đồ thị

```text
q = deque([start]); visited = {start}
while q:
    u = q.popleft()
    xử lý u
    for v in hàng_xóm(u):
        if v not in visited:
            visited.add(v)        # đánh dấu NGAY khi đưa vào hàng đợi
            q.append(v)
```

In [3]:
from collections import deque

do_thi = {                       # danh sách kề
    "A": ["B", "C"], "B": ["A", "D", "E"], "C": ["A", "F"],
    "D": ["B"], "E": ["B", "F"], "F": ["C", "E"],
}

def bfs(graph, start):
    q, visited, thu_tu = deque([start]), {start}, []
    while q:
        u = q.popleft()
        thu_tu.append(u)
        for v in graph[u]:
            if v not in visited:
                visited.add(v)
                q.append(v)
        print(f"lấy {u} | hàng đợi: {list(q)}")
    return thu_tu

print("Thứ tự BFS:", bfs(do_thi, "A"))

lấy A | hàng đợi: ['B', 'C']
lấy B | hàng đợi: ['C', 'D', 'E']
lấy C | hàng đợi: ['D', 'E', 'F']
lấy D | hàng đợi: ['E', 'F']
lấy E | hàng đợi: ['F']
lấy F | hàng đợi: []
Thứ tự BFS: ['A', 'B', 'C', 'D', 'E', 'F']


### BFS theo tầng (level-order) trên cây nhị phân

Muốn biết nút nào thuộc tầng nào: ở đầu mỗi vòng, hàng đợi chứa **đúng** các nút của một tầng. Ghi lại `len(q)`, lấy ra đúng chừng ấy nút (và đẩy con của chúng vào) — đó là một tầng.

In [4]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

cay_bfs = TreeNode(3, TreeNode(9), TreeNode(20, TreeNode(15), TreeNode(7)))

def theo_tang(root):
    if not root:
        return []
    q, tang = deque([root]), []
    while q:
        cur = []
        for _ in range(len(q)):          # đúng số nút của tầng hiện tại
            node = q.popleft()
            cur.append(node.val)
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
        tang.append(cur)
    return tang

print("các tầng:", theo_tang(cay_bfs))

các tầng: [[3], [9, 20], [15, 7]]


### BFS tìm đường ngắn nhất trên lưới

Trên đồ thị **không trọng số** (mỗi bước dài như nhau), lần đầu BFS chạm tới một ô chính là đi bằng **số bước ít nhất**. Lưu `dist` khi đưa ô vào hàng đợi.

In [5]:
luoi = ["S.#.",
        "..#.",
        "...E"]

def ngan_nhat(grid):
    R, C = len(grid), len(grid[0])
    start = next((r, c) for r in range(R) for c in range(C) if grid[r][c] == "S")
    dist = {start: 0}
    q = deque([start])
    while q:
        r, c = q.popleft()
        if grid[r][c] == "E":
            return dist[(r, c)]
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] != "#" and (nr, nc) not in dist:
                dist[(nr, nc)] = dist[(r, c)] + 1
                q.append((nr, nc))
    return -1

print("số bước ít nhất từ S tới E:", ngan_nhat(luoi))

số bước ít nhất từ S tới E: 5


### Độ phức tạp và lỗi hay gặp

`O(V + E)` thời gian (mỗi đỉnh vào hàng đợi một lần, mỗi cạnh xét một/hai lần), `O(V)` bộ nhớ. Trên cây: `O(n)`; trên lưới `R×C`: `O(R·C)`.

- Đánh dấu `visited` khi **lấy ra** thay vì khi **đưa vào** → một ô có thể vào hàng đợi nhiều lần (chậm, đếm sai).
- Dùng `list.pop(0)` → mỗi lần `O(n)`.
- Dùng `pop()` (cuối) → thành DFS, mất tính "gần trước".
- Quên kiểm tra biên lưới.

## 2.3 Áp dụng vào bài này

Có thể xem mỗi chỉ số là một đỉnh, nhảy được tới các chỉ số trong tầm → số bước ít nhất = **BFS** theo tầng. Lời giải `O(n)` là **tham lam** "BFS ngầm": tầng hiện tại là đoạn `[start, end]`, tầng kế là mọi chỉ số với tới được từ tầng này (`farthest`), không cần hàng đợi.

# 3. Cách 1: Quy hoạch động

Gọi `dp[i]` = số lần nhảy ít nhất để tới `i`. `dp[0] = 0`.

Từ mỗi `i` đã tới được, cập nhật mọi vị trí `i + 1 .. i + nums[i]`:

```python
dp[k] = min(dp[k], dp[i] + 1)
```

In [6]:
from typing import List

def jump_dp(nums: List[int]) -> int:
    n = len(nums)
    INF = float("inf")
    dp = [INF] * n
    dp[0] = 0
    for i in range(n):
        if dp[i] == INF:
            continue
        for k in range(i + 1, min(n, i + nums[i] + 1)):
            dp[k] = min(dp[k], dp[i] + 1)
    return dp[-1]

print(jump_dp([2, 3, 1, 1, 4]))
print(jump_dp([2, 3, 0, 1, 4]))

2
2


**Độ phức tạp:** `O(n · max(nums))` — tới `10^4 × 1000 = 10^7` phép tính. Chạy được nhưng chậm, và trên LeetCode Python dễ bị chậm. Ta có thể làm `O(n)`.

# 4. Quan sát then chốt: các "tầng" là đoạn liên tiếp

Gọi **tầng `k`** là tập vị trí có `dp = k` (tới được bằng đúng `k` bước, không ít hơn).

Với `nums = [2, 3, 1, 1, 4]`:

In [7]:
nums = [2, 3, 1, 1, 4]
n = len(nums)
dp = [float("inf")] * n
dp[0] = 0
for i in range(n):
    for k in range(i + 1, min(n, i + nums[i] + 1)):
        dp[k] = min(dp[k], dp[i] + 1)
print("dp =", dp)

dp = [0, 1, 1, 2, 2]


```text
chỉ số:  0 | 1  2 | 3  4
tầng:    0 | 1  1 | 2  2
```

- Tầng 0 = `[0, 0]`.
- Tầng 1 = `[1, 2]` — mọi vị trí tới được từ tầng 0 mà chưa thuộc tầng trước.
- Tầng 2 = `[3, 4]`.

**Mỗi tầng là một đoạn liên tiếp**, vì: nếu tới được vị trí `p` thì cũng tới được mọi vị trí giữa nguồn và `p` (nhảy ngắn hơn luôn được phép).

Vậy:

- Tầng `k + 1` bắt đầu ngay sau tầng `k`.
- Tầng `k + 1` kết thúc ở **vị trí xa nhất** mà bất kỳ ô nào trong tầng `k` nhảy tới được: `max(i + nums[i])` với `i` thuộc tầng `k`.

Đây chính là **BFS** (duyệt theo chiều rộng), nhưng vì mỗi tầng là đoạn liên tiếp, ta chỉ cần lưu **điểm cuối** của tầng — không cần hàng đợi.

# 5. Lời giải Greedy

Hai biến:

- `current_end`: điểm cuối của tầng **hiện tại**.
- `farthest`: điểm xa nhất tới được từ các vị trí đã duyệt (sẽ là điểm cuối tầng **kế tiếp**).

Duyệt `i` từ `0` tới `n - 2`:

1. `farthest = max(farthest, i + nums[i])`.
2. Nếu `i == current_end` (đã duyệt hết tầng hiện tại) → phải nhảy thêm một lần: `jumps += 1`, `current_end = farthest`.

Vì sao duyệt tới `n - 2`? Khi đã tới `n - 1` thì không cần nhảy nữa. Nếu duyệt tới `n - 1`, trường hợp `current_end == n - 1` sẽ đếm thừa một lần nhảy.

In [8]:
def jump_greedy(nums: List[int]) -> int:
    jumps = 0
    current_end = 0
    farthest = 0
    for i in range(len(nums) - 1):
        farthest = max(farthest, i + nums[i])
        if i == current_end:
            jumps += 1
            current_end = farthest
            if current_end >= len(nums) - 1:
                break
    return jumps

print(jump_greedy([2, 3, 1, 1, 4]))
print(jump_greedy([2, 3, 0, 1, 4]))
print(jump_greedy([0]))

2
2
0


`break` sớm khi tầng mới đã chạm đích — không bắt buộc nhưng tránh vòng lặp thừa.

# 6. Mô phỏng từng bước

`nums = [2, 3, 1, 1, 4]`

| i | nums[i] | i + nums[i] | farthest | i == current_end? | jumps | current_end |
|---|---|---|---|---|---|---|
| 0 | 2 | 2 | 2 | có (0) | 1 | 2 |
| 1 | 3 | 4 | 4 | không | 1 | 2 |
| 2 | 1 | 3 | 4 | có (2) | 2 | 4 → chạm đích, dừng |

Kết quả `2`.

In [9]:
def trace_jump(nums: List[int]) -> int:
    jumps, current_end, farthest = 0, 0, 0
    print(f"nums = {nums}")
    for i in range(len(nums) - 1):
        farthest = max(farthest, i + nums[i])
        line = f"i={i} nums[i]={nums[i]} -> farthest={farthest}"
        if i == current_end:
            jumps += 1
            print(f"{line} | hết tầng (i == {current_end}) -> nhảy lần {jumps}, tầng mới kết thúc ở {farthest}")
            current_end = farthest
            if current_end >= len(nums) - 1:
                print("  tầng mới đã chứa đích -> dừng")
                break
        else:
            print(line)
    print("Kết quả:", jumps)
    return jumps

_ = trace_jump([2, 3, 1, 1, 4])
print()
_ = trace_jump([1, 2, 1, 1, 1])

nums = [2, 3, 1, 1, 4]
i=0 nums[i]=2 -> farthest=2 | hết tầng (i == 0) -> nhảy lần 1, tầng mới kết thúc ở 2
i=1 nums[i]=3 -> farthest=4
i=2 nums[i]=1 -> farthest=4 | hết tầng (i == 2) -> nhảy lần 2, tầng mới kết thúc ở 4
  tầng mới đã chứa đích -> dừng
Kết quả: 2

nums = [1, 2, 1, 1, 1]
i=0 nums[i]=1 -> farthest=1 | hết tầng (i == 0) -> nhảy lần 1, tầng mới kết thúc ở 1
i=1 nums[i]=2 -> farthest=3 | hết tầng (i == 1) -> nhảy lần 2, tầng mới kết thúc ở 3
i=2 nums[i]=1 -> farthest=3
i=3 nums[i]=1 -> farthest=4 | hết tầng (i == 3) -> nhảy lần 3, tầng mới kết thúc ở 4
  tầng mới đã chứa đích -> dừng
Kết quả: 3


# 7. Phiên bản đúng format LeetCode

In [10]:
class Solution:
    def jump(self, nums: List[int]) -> int:
        jumps = 0
        current_end = 0
        farthest = 0
        for i in range(len(nums) - 1):
            farthest = max(farthest, i + nums[i])
            if i == current_end:
                jumps += 1
                current_end = farthest
                if current_end >= len(nums) - 1:
                    break
        return jumps

In [11]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<26} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.jump([2, 3, 1, 1, 4]), 2)
check("Ví dụ 2", sol.jump([2, 3, 0, 1, 4]), 2)
check("Một phần tử", sol.jump([0]), 0)
check("Hai phần tử", sol.jump([1, 0]), 1)
check("Nhảy 1 lần tới đích", sol.jump([5, 0, 0, 0, 0]), 1)
check("Toàn bước 1", sol.jump([1, 1, 1, 1]), 3)
check("Dài hơn", sol.jump([2, 3, 1, 1, 4, 0, 0]), 3)

Ví dụ 1                    PASS  2
Ví dụ 2                    PASS  2
Một phần tử                PASS  0
Hai phần tử                PASS  1
Nhảy 1 lần tới đích        PASS  1
Toàn bước 1                PASS  3
Dài hơn                    PASS  3


In [12]:
import random

def random_reachable(n):
    """Sinh mảng luôn tới được đích: đảm bảo farthest không bị kẹt."""
    while True:
        nums = [random.randint(0, 4) for _ in range(n)]
        far = 0
        ok = True
        for i in range(n - 1):
            if i > far:
                ok = False
                break
            far = max(far, i + nums[i])
        if ok and far >= n - 1 or n == 1:
            return nums

random.seed(45)
for _ in range(3000):
    nums = random_reachable(random.randint(1, 15))
    assert sol.jump(nums) == jump_dp(nums), nums
print("3000 test ngẫu nhiên: Greedy khớp DP")

3000 test ngẫu nhiên: Greedy khớp DP


# 8. Vì sao Greedy đúng? Độ phức tạp?

**Bất biến:** khi vòng lặp xử lý xong mọi `i` thuộc tầng `k` (tức `i` chạy tới `current_end` của tầng `k`):

- `jumps = k + 1`,
- `current_end` = điểm cuối của tầng `k + 1` = vị trí xa nhất tới được bằng `k + 1` bước.

Vì mỗi tầng là đoạn liên tiếp (mục 4) và tầng `k + 1` kết thúc đúng ở `max(i + nums[i])` trên tầng `k`, bất biến được duy trì. Lần đầu tiên tầng chứa `n - 1`, `jumps` là số bước ít nhất — giống BFS.

**Khác với greedy "luôn nhảy xa nhất có thể":** ta **không** quyết định nhảy tới đâu ở mỗi bước; ta chỉ đếm số tầng. Cách "luôn nhảy xa nhất" có thể sai:

In [13]:
def wrong_always_max(nums: List[int]) -> int:
    i, jumps = 0, 0
    while i < len(nums) - 1:
        i = min(len(nums) - 1, i + nums[i])      # SAI: luôn nhảy tối đa
        jumps += 1
        if nums[i] == 0 and i < len(nums) - 1:
            return -1                           # kẹt
    return jumps

print("Luôn nhảy xa nhất:", wrong_always_max([2, 3, 1, 1, 4]), "| Đúng:", sol.jump([2, 3, 1, 1, 4]))

Luôn nhảy xa nhất: 3 | Đúng: 2


**Độ phức tạp:**

```text
Time:  O(n)    — một lần duyệt
Space: O(1)
```

| Cách | Thời gian | Bộ nhớ |
|---|---|---|
| DP | O(n · max(nums)) | O(n) |
| BFS với hàng đợi | O(n · max(nums)) | O(n) |
| Greedy theo tầng | O(n) | O(1) |

# 9. Những lỗi dễ mắc

## Lỗi 1: Luôn nhảy xa nhất từ vị trí hiện tại

Ví dụ `[2, 3, 1, 1, 4]`: từ `0` nhảy xa nhất tới `2` (giá trị 1), rồi `3`, rồi `4` → 3 lần. Tối ưu là `0 → 1 → 4` (2 lần). Điểm nhảy tốt nhất là điểm **mở ra tầng xa nhất**, không phải điểm **xa nhất**.

## Lỗi 2: Duyệt tới `n - 1` thay vì `n - 2`

In [14]:
def wrong_loop_to_end(nums: List[int]) -> int:
    jumps, current_end, farthest = 0, 0, 0
    for i in range(len(nums)):                    # SAI: tới n - 1
        farthest = max(farthest, i + nums[i])
        if i == current_end:
            jumps += 1
            current_end = farthest
    return jumps

print("Sai :", wrong_loop_to_end([0]), wrong_loop_to_end([2, 3, 1, 1, 4]))
print("Đúng:", sol.jump([0]), sol.jump([2, 3, 1, 1, 4]))

Sai : 1 3
Đúng: 0 2


Với `[0]`: đã ở đích, cần 0 bước, nhưng bản sai đếm 1. Với `[2, 3, 1, 1, 4]`: khi `i = 4 = current_end`, bản sai đếm thêm một lần nhảy thừa.

## Lỗi 3: Cập nhật `farthest` sau khi kiểm tra `i == current_end`

Phải cập nhật `farthest` với vị trí `i` **trước**, vì `i` cũng thuộc tầng hiện tại và có thể mở rộng tầng kế tiếp.

## Lỗi 4: Quên trường hợp `n = 1`

Đã ở đích → 0 bước. Vòng `range(0)` không chạy, trả `0` — đúng tự nhiên.

# 10. Mẹo nhận diện pattern

- "Số bước **ít nhất**" trên đồ thị không trọng số → **BFS**.
- Nếu các tầng BFS là **đoạn liên tiếp** trên mảng → thay hàng đợi bằng **hai biến biên** → Greedy `O(n)`.
- "Có tới được không?" (không hỏi số bước) → chỉ cần `farthest` (**#55 Jump Game**).

Bài liên quan: **#55 Jump Game**, **#1306 Jump Game III** (BFS thật), **#1024 Video Stitching**, **#1326 Minimum Number of Taps** — cùng dạng "phủ đoạn bằng ít khoảng nhất".

## Template

```python
jumps = current_end = farthest = 0
for i in range(n - 1):
    farthest = max(farthest, i + nums[i])
    if i == current_end:
        jumps += 1
        current_end = farthest
return jumps
```

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Chia `nums = [1, 2, 1, 1, 1]` thành các tầng. Đáp án là bao nhiêu?

## Bài 2 — Jump Game (LeetCode #55)

Trả về `True` nếu tới được chỉ số cuối (không đảm bảo luôn tới được).

```python
can_jump([2, 3, 1, 1, 4]) -> True
can_jump([3, 2, 1, 0, 4]) -> False
```

In [15]:
def exercise_2_can_jump(nums: List[int]) -> bool:
    # TODO: duy trì farthest; nếu i > farthest thì kẹt
    pass


# assert exercise_2_can_jump([2, 3, 1, 1, 4]) is True
# assert exercise_2_can_jump([3, 2, 1, 0, 4]) is False
# assert exercise_2_can_jump([0]) is True

## Bài 3 — Điền chỗ trống

In [16]:
def exercise_3_jump(nums: List[int]) -> int:
    jumps = current_end = farthest = 0
    for i in range(len(nums) - 1):
        # TODO 1: cập nhật farthest
        # TODO 2: nếu hết tầng hiện tại -> tăng jumps, chuyển current_end
        pass
    return jumps


# assert exercise_3_jump([2, 3, 1, 1, 4]) == 2
# assert exercise_3_jump([0]) == 0
# assert exercise_3_jump([1, 1, 1, 1]) == 3

## Bài 4 — Biến thể: trả về đường đi

Trả về **danh sách chỉ số** của một đường đi ngắn nhất từ `0` tới `n - 1`.

```python
jump_path([2, 3, 1, 1, 4]) -> [0, 1, 4]
jump_path([0])             -> [0]
```

Gợi ý nhanh: trong mỗi tầng, nhớ vị trí `i` cho ra `farthest` lớn nhất — đó là điểm "đạp nhảy".

In [17]:
def exercise_4_jump_path(nums: List[int]) -> List[int]:
    # TODO
    pass


# assert exercise_4_jump_path([2, 3, 1, 1, 4]) == [0, 1, 4]
# assert exercise_4_jump_path([0]) == [0]
# assert len(exercise_4_jump_path([1, 1, 1, 1])) == 4

## Bài 5 — Mini challenge: Jump Game III (LeetCode #1306)

Từ `i` chỉ được nhảy tới **đúng** `i + arr[i]` hoặc `i - arr[i]`. Bắt đầu ở `start`, kiểm tra có tới được **ô nào có giá trị 0** không.

```python
can_reach([4, 2, 3, 0, 3, 1, 2], 5) -> True
can_reach([3, 0, 2, 1, 2], 2)       -> False
```

Ở đây các tầng **không còn liên tiếp** → cần BFS thật (hàng đợi + tập `visited`).

In [18]:
def exercise_5_can_reach(arr: List[int], start: int) -> bool:
    # TODO
    pass


# assert exercise_5_can_reach([4, 2, 3, 0, 3, 1, 2], 5) is True
# assert exercise_5_can_reach([4, 2, 3, 0, 3, 1, 2], 0) is True
# assert exercise_5_can_reach([3, 0, 2, 1, 2], 2) is False

# 12. Gợi ý

**Bài 1**
- Mức 1: tầng 0 = `[0]`; từ 0 tới được tối đa chỉ số 1.

**Bài 2**
- Mức 1: `farthest = 0`; với mỗi `i`: nếu `i > farthest` → `False`.
- Mức 2: cập nhật `farthest = max(farthest, i + nums[i])`.

**Bài 4**
- Mức 1: lưu thêm `best_i` = vị trí cho `farthest` lớn nhất trong tầng hiện tại.
- Mức 2: khi hết tầng, thêm `best_i` vào đường đi (trừ khi nó là vị trí đã có).

**Bài 5**
- Mức 1: `from collections import deque`; `queue = deque([start])`, `seen = {start}`.
- Mức 2: lấy `i` ra, nếu `arr[i] == 0` → `True`; thêm `i ± arr[i]` hợp lệ chưa thăm.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

Tầng 0 = `[0]`, tầng 1 = `[1]`, tầng 2 = `[2, 3]`, tầng 3 = `[4]` → đáp án `3`.

In [19]:
_ = trace_jump([1, 2, 1, 1, 1])

nums = [1, 2, 1, 1, 1]
i=0 nums[i]=1 -> farthest=1 | hết tầng (i == 0) -> nhảy lần 1, tầng mới kết thúc ở 1
i=1 nums[i]=2 -> farthest=3 | hết tầng (i == 1) -> nhảy lần 2, tầng mới kết thúc ở 3
i=2 nums[i]=1 -> farthest=3
i=3 nums[i]=1 -> farthest=4 | hết tầng (i == 3) -> nhảy lần 3, tầng mới kết thúc ở 4
  tầng mới đã chứa đích -> dừng
Kết quả: 3


## Đáp án Bài 2

In [20]:
def answer_2_can_jump(nums: List[int]) -> bool:
    farthest = 0
    for i, x in enumerate(nums):
        if i > farthest:
            return False
        farthest = max(farthest, i + x)
    return True

assert answer_2_can_jump([2, 3, 1, 1, 4]) is True
assert answer_2_can_jump([3, 2, 1, 0, 4]) is False
assert answer_2_can_jump([0]) is True
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [21]:
def answer_3_jump(nums: List[int]) -> int:
    jumps = current_end = farthest = 0
    for i in range(len(nums) - 1):
        farthest = max(farthest, i + nums[i])
        if i == current_end:
            jumps += 1
            current_end = farthest
    return jumps

assert answer_3_jump([2, 3, 1, 1, 4]) == 2
assert answer_3_jump([0]) == 0
assert answer_3_jump([1, 1, 1, 1]) == 3
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [22]:
def answer_4_jump_path(nums: List[int]) -> List[int]:
    n = len(nums)
    path = [0]
    current_end = farthest = 0
    best_i = 0
    for i in range(n - 1):
        if i + nums[i] > farthest:
            farthest = i + nums[i]
            best_i = i
        if i == current_end:
            if best_i != path[-1]:
                path.append(best_i)
            current_end = farthest
            if current_end >= n - 1:
                break
    if path[-1] != n - 1:
        path.append(n - 1)
    return path

assert answer_4_jump_path([2, 3, 1, 1, 4]) == [0, 1, 4]
assert answer_4_jump_path([0]) == [0]
assert len(answer_4_jump_path([1, 1, 1, 1])) == 4
for _ in range(2000):
    nums = random_reachable(random.randint(1, 12))
    p = answer_4_jump_path(nums)
    assert p[0] == 0 and p[-1] == len(nums) - 1
    assert all(0 < p[k + 1] - p[k] <= nums[p[k]] for k in range(len(p) - 1)), (nums, p)
    assert len(p) - 1 == jump_dp(nums), (nums, p)
print("Bài 4 PASS (đường đi hợp lệ và ngắn nhất trên 2000 test)")

Bài 4 PASS (đường đi hợp lệ và ngắn nhất trên 2000 test)

## Đáp án Bài 5

In [23]:
from collections import deque

def answer_5_can_reach(arr: List[int], start: int) -> bool:
    queue = deque([start])
    seen = {start}
    while queue:
        i = queue.popleft()
        if arr[i] == 0:
            return True
        for nxt in (i + arr[i], i - arr[i]):
            if 0 <= nxt < len(arr) and nxt not in seen:
                seen.add(nxt)
                queue.append(nxt)
    return False

assert answer_5_can_reach([4, 2, 3, 0, 3, 1, 2], 5) is True
assert answer_5_can_reach([4, 2, 3, 0, 3, 1, 2], 0) is True
assert answer_5_can_reach([3, 0, 2, 1, 2], 2) is False
print("Bài 5 PASS")

Bài 5 PASS


# 14. Tổng kết

> Các vị trí tới được bằng đúng `k` bước tạo thành **một đoạn liên tiếp** (một "tầng"). Duyệt mảng, giữ `farthest`; mỗi khi đi hết tầng hiện tại (`i == current_end`), tăng số bước và chuyển sang tầng mới (`current_end = farthest`).

---

## Checklist trước khi nộp

- [ ] Vòng lặp tới `n - 2` (`range(len(nums) - 1)`)?
- [ ] Cập nhật `farthest` **trước** khi kiểm tra `i == current_end`?
- [ ] Không dùng greedy "luôn nhảy xa nhất"?
- [ ] `n = 1` trả về `0`?